**Reading CSV Files into PySpark DataFrames**

In [0]:
customer_df=spark.read.table("workspace.default.customers")
display(customer_df)
orders_df=spark.read.table("workspace.default.orders")
product_df=spark.read.table("workspace.default.products")
return_df=spark.read.table("workspace.default.returns")
payment_df=spark.read.table("workspace.default.payments")

customer_id,customer_name,city,state,signup_date
1,Frank Boyd,Martinmouth,Oklahoma,2025-10-27
2,Lance Parker,Clarkville,North Dakota,2026-05-03
3,Brandon Baker,North Ronald,Michigan,2023-12-29
4,Linda Taylor,Elaineton,West Virginia,2025-03-17
5,Michael Leon,Karenland,Connecticut,2025-02-05
6,Michelle Hahn,Martinborough,Montana,2025-03-01
7,Tonya Gibson,North Isaiah,Oklahoma,2023-08-23
8,William Garner,Copelandfort,Texas,2023-07-29
9,David Gentry,Obrienton,Oregon,2025-01-14
10,Dana Murphy,Campbellborough,Virginia,2024-08-11


**Displaying Schema Information**

In [0]:
customer_df.printSchema()
orders_df.printSchema()
product_df.printSchema()
payment_df.printSchema()
return_df.printSchema()

root
 |-- customer_id: long (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- signup_date: date (nullable = true)

root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- quantity: long (nullable = true)
 |-- sales: double (nullable = true)
 |-- order_date: date (nullable = true)

root
 |-- product_id: long (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)

root
 |-- payment_id: long (nullable = true)
 |-- order_id: long (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- payment_status: string (nullable = true)

root
 |-- return_id: long (nullable = true)
 |-- order_id: long (nullable = true)
 |-- return_reason: string (nullable = true)



**Previewing DataFrames**

In [0]:
customer_df.show(5)
orders_df.show(5)
product_df.show(5)
payment_df.show(5)
return_df.show(5)

+-----------+-------------+------------+-------------+-----------+
|customer_id|customer_name|        city|        state|signup_date|
+-----------+-------------+------------+-------------+-----------+
|          1|   Frank Boyd| Martinmouth|     Oklahoma| 2025-10-27|
|          2| Lance Parker|  Clarkville| North Dakota| 2026-05-03|
|          3|Brandon Baker|North Ronald|     Michigan| 2023-12-29|
|          4| Linda Taylor|   Elaineton|West Virginia| 2025-03-17|
|          5| Michael Leon|   Karenland|  Connecticut| 2025-02-05|
+-----------+-------------+------------+-------------+-----------+
only showing top 5 rows
+--------+-----------+----------+--------+------+----------+
|order_id|customer_id|product_id|quantity| sales|order_date|
+--------+-----------+----------+--------+------+----------+
|       1|        116|         3|       1|918.97|2025-11-15|
|       2|        104|        22|       2|570.73|2026-01-21|
|       3|        186|        18|       3|267.58|2026-02-27|
|      

**Null Value Handling Transformation**

In [0]:
order_clean_df=orders_df.fillna({"sales":0})

**Duplicate Record Removal Transformation**

In [0]:
order_df=order_clean_df.dropDuplicates()

**Data Type Conversion Transformation**

In [0]:
from pyspark.sql.functions import *

order_df1=order_df.withColumn("sales",col("sales").cast("double"))
order_df1.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- quantity: long (nullable = true)
 |-- sales: double (nullable = false)
 |-- order_date: date (nullable = true)



**Revenue Transformation & Derived Column Creation** 

In [0]:
order_df2=order_df1.withColumn("Total_revanue",col("quantity")*col("sales"))
order_df2.show(5)

+--------+-----------+----------+--------+------+----------+-------------+
|order_id|customer_id|product_id|quantity| sales|order_date|Total_revanue|
+--------+-----------+----------+--------+------+----------+-------------+
|       1|        116|         3|       1|918.97|2025-11-15|       918.97|
|       2|        104|        22|       2|570.73|2026-01-21|      1141.46|
|       3|        186|        18|       3|267.58|2026-02-27|       802.74|
|       4|          3|        37|       1|855.57|2026-02-12|       855.57|
|       5|         76|         4|       1|179.14|2025-10-02|       179.14|
+--------+-----------+----------+--------+------+----------+-------------+
only showing top 5 rows


**Payment Status Aggregation Transformation**

In [0]:
payment_status_df = payment_df.groupBy(
    "payment_status"
).count()

payment_status_df.show()

+--------------+-----+
|payment_status|count|
+--------------+-----+
|     Completed|  333|
|       Pending|  345|
|        Failed|  322|
+--------------+-----+

